In [ ]:
# Check environment
import os
import sys

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    print("Running on Kaggle")
    # Kaggle paths
    TRAIN_FILE = '/kaggle/input/arabic-diacritization-dataset/train.txt'
    DEV_FILE = '/kaggle/input/arabic-diacritization-dataset/dev.txt'
    TEST_FILE = '/kaggle/input/arabic-diacritization-dataset/test.txt'
    OUTPUT_FILE = '/kaggle/working/submission.csv'
else:
    print("Running locally")
    # Local paths
    TRAIN_FILE = 'data/train.txt'
    DEV_FILE = 'data/val.txt'
    TEST_FILE = 'data/test.txt'
    OUTPUT_FILE = 'submission.csv'

print(f"Train file: {TRAIN_FILE}")
print(f"Dev file: {DEV_FILE}")
print(f"Test file: {TEST_FILE}")
print(f"Output file: {OUTPUT_FILE}")

## Import Training Module

All the heavy implementation is in the `/src` directory. This notebook just calls the training function.

In [ ]:
# Import training function
from src.training.train_crf import run_crf_training

print("✓ Imports successful")

## Train CRF Model

This will:
1. Load and preprocess data
2. Extract rich contextual features for CRF
3. Train CRF using forward-backward algorithm
4. Evaluate on dev set with Viterbi decoding
5. Generate predictions for test set
6. Save predictions to CSV

**Note**: CRF training may take 15-30 minutes due to forward-backward computations.

In [ ]:
# Run complete training pipeline
run_crf_training(
    train_file=TRAIN_FILE,
    dev_file=DEV_FILE,
    test_file=TEST_FILE,
    output_file=OUTPUT_FILE
)

print("\n✓ Training and prediction completed!")
print(f"✓ Submission file saved to: {OUTPUT_FILE}")

## Verify Submission File

Let's check the first few lines of the submission file.

In [ ]:
import csv

print("Submission file preview:")
print("-" * 80)

with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader):
        if i < 10:  # Show first 10 lines
            print(f"{row[0]}: {row[1][:50]}..." if len(row[1]) > 50 else f"{row[0]}: {row[1]}")
        else:
            break

print("-" * 80)
print("✓ Submission file is ready for upload!")

## Model Details

### CRF Architecture
- **Type**: Linear-chain CRF for sequence labeling
- **Features**: Contextual character features
  - Current character
  - Previous 1-2 characters  
  - Next 1-2 characters
  - Character bigrams and trigrams
  - Character type (letter/space/digit)
  - Word boundaries (start/end)
- **Transitions**: Learnable transition scores between all label pairs

### Training Algorithm
1. **Forward Algorithm**: Compute forward probabilities (α) in log-space
2. **Backward Algorithm**: Compute backward probabilities (β) in log-space
3. **Gradient Computation**: 
   - Expected feature counts from model (using α, β)
   - Observed feature counts from data
   - Gradient = observed - expected
4. **Parameter Update**: Gradient ascent with L2 regularization

### Inference
- **Viterbi Decoding**: Find most likely label sequence
- Dynamic programming to find optimal path

### Parameters
- Learning rate: 0.01
- Max iterations: 50
- L2 penalty: 0.1

### Implementation
All code is written from scratch using only NumPy:
- Forward-backward algorithm in log-space
- Gradient computation for CRF
- Viterbi decoding
- Feature extraction from character sequences

**No external ML libraries used!**